# 文献调查：研究论文摘要器

做文献调查时，往往要翻几十篇 PDF，才能判断哪些真正和主题相关——可能花费数小时甚至数天。本工具自动分析每个 PDF、抽取关键字段，并生成带**相关性分数**与**颜色编码**的结构化 Excel，让你立刻看出哪些值得精读。

## 这个工具做什么
- 读取文件夹中的全部 PDF 研究论文
- 抽取关键字段：目标、方法、验证方法、标准、局限等
- 按与你研究主题的相关性打分（1–10）
- 生成颜色编码 Excel：绿色（高相关）、黄色（中等）、红色（低相关）
- 自动按相关性从高到低排序

## 和本课概念的关系
| 概念 | 本笔记本里 |
|------|------------|
| Chat Completions + system/user | `system_prompt` 定抽取 schema，user 塞入论文全文 |
| 结构化输出（JSON） | 要求模型只回合法 JSON，再用 `json.loads` 解析 |
| 业务落地 | PDF → 表格 → 条件格式，典型「调研加速」用例 |

## 怎么用
1. 把 PDF 放进与本笔记本同目录的 `papers/` 文件夹
2. 把下面的 `research_topic` 改成你的研究领域（主题字符串若影响打分语义则保持英文）
3. 依次运行所有单元格
4. 在同目录查看生成的 Excel 输出文件

## 要求
- `.env` 中配置 OpenAI API Key
- 依赖：`uv pip install pdfplumber pandas openpyxl openai python-dotenv`

---
*作为 Udemy 课程第一天扩展项目构建：AI 工程师核心课程 — LLM 工程、RAG、QLoRA、Agents（Ed Donner）。*


### 安装所需的库

运行下面安装单元格后再跑其余部分：

- `pdfplumber` — 从 PDF 提取文本
- `pandas` — 创建与操作数据表（DataFrame）
- `openpyxl` — 创建并格式化 Excel
- `openai` — 连接 OpenAI API 分析论文
- `python-dotenv` — 从 `.env` 安全加载 API Key


In [ ]:
# 用 uv 一次性安装本笔记本依赖（魔法命令，保持原样）
!uv pip install pdfplumber pandas openpyxl openai python-dotenv


In [ ]:
# ========== 导入：标准库 + 第三方工具 ==========

# 标准库 os：拼路径、列出文件夹里的文件名
import os
# 标准库 json：把模型返回的 JSON 字符串解析成 Python 字典
import json

# pdfplumber：从 PDF 每一页抽出文本
import pdfplumber          # extracts text from PDF files
# pandas：把多篇论文的字典结果收成表格，并导出 Excel
import pandas as pd        # creates and manipulates the data table
# OpenAI 客户端：后面用 chat.completions.create 调模型
from openai import OpenAI  # connects to the OpenAI API
# openpyxl：打开已写出的 xlsx，以便继续改样式
from openpyxl import load_workbook                    # opens and edits Excel files
# PatternFill / Alignment：单元格底色与对齐（换行、顶对齐）
from openpyxl.styles import PatternFill, Alignment    # formats Excel cells
# load_dotenv：从 .env 读入 OPENAI_API_KEY 等，避免密钥写进笔记本
from dotenv import load_dotenv                        # loads API key from .env file


In [ ]:
# ========== 加载环境变量：把 .env 里的 OpenAI Key 读进进程 ==========

# override=True：用 .env 覆盖已有同名环境变量
load_dotenv(override=True)


In [ ]:
# ========== 初始化 OpenAI 客户端：后续所有 API 调用都走它 ==========

# 默认从环境变量 OPENAI_API_KEY 取密钥（由上一格 load_dotenv 注入）
openai = OpenAI()


In [ ]:
# ========== 主流程：定主题 → 抽 PDF 文本 → LLM 出 JSON → 表格 → 着色 Excel ==========

# 改成你的研究主题（发给模型的主题描述保持英文，以便相关性量表语义不变）
research_topic = """
UAV intelligence quality assurance, standards, and validation methods. 
Topics of interest include: UAV system reliability, fault detection, 
testing frameworks, quality standards, validation methodologies, 
and intelligent UAV systems.
"""

# 系统提示：告诉 LLM 扮演文献助手、必须抽哪些字段、只许回合法 JSON
# 用 f-string 把 research_topic 嵌进提示，便于按「你的主题」打相关性分
# JSON 花括号在 f-string 里要写成 {{ }}，运行时才变成单个 { }

system_prompt = f"""
You are a research assistant helping with a literature survey on this topic: {research_topic}

Extract the following fields from the research paper and respond ONLY in valid JSON format with these exact keys:
{{
    "Year": "Look carefully for the publication year in the copyright notice, journal header, submission date, or first page. Return only the 4-digit year. If truly not found, write Unknown",
    "Paper Title": "",
    "Authors": "",
    "Application Domain": "",
    "AI / Intelligence Component": "",
    "Objective": "",
    "Validation Method": "",
    "Test Environment": "",
    "Evaluation Metrics": "",
    "Robustness / Safety Testing": "",
    "Standards Mentioned": "",
    "Standards Body Referenced": "",
    "Limitations": "",
    "Is Relevant": "Yes or No only",
    "Relevance Score": "Rate strictly from 1 to 10 based on how directly the paper addresses UAV intelligence quality assurance, validation standards, or certification methods. 9-10: paper directly addresses UAV QA frameworks, validation standards, or certification as its PRIMARY contribution. 7-8: paper covers UAV fault detection, reliability, or safety testing but QA/standards is not the main focus. 5-6: paper uses UAVs as a tool for another application like inspection, agriculture, or mapping with minimal QA focus. 1-4: paper has little or no connection to UAV QA or validation standards."
}}
"""

# ---------- 从 PDF 每一页抽取文本并拼接 ----------
# 若模型上下文窗口较小，可改为 return text[:15000] 截断（原注释提示，勿擅自改逻辑）
def extract_text_from_pdf(pdf_path):
    # with：打开 PDF，离开块时自动关闭文件句柄
    with pdfplumber.open(pdf_path) as pdf:
        # 逐页 extract_text；空页（None）跳过，再用换行拼成一整篇
        text = "\n".join(page.extract_text() for page in pdf.pages if page.extract_text())
    return text  

# ---------- 单篇论文：抽文本 → 调 Chat Completions → 解析 JSON 字典 ----------
def analyze_paper(pdf_path):
    # 先把 PDF 变成纯文本
    text = extract_text_from_pdf(pdf_path)
    # messages：system 定 schema，user 放入论文正文（提示字符串保持英文）
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Here is the research paper:\n\n{text}"}
    ]
    # 调用 OpenAI；model id 保持原样（gpt-5-nano）
    response = openai.chat.completions.create(
        model="gpt-5-nano",   # fastest and cheapest GPT-5 model, great for summarization
        messages=messages
    )
    # 取出助手回复的原始字符串
    raw = response.choices[0].message.content
    # 去掉模型可能包在外面的 ```json ... ``` 围栏，再 json.loads
    raw = raw.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(raw)


# 论文文件夹路径（相对本笔记本）；把 PDF 放进此目录
papers_folder = "papers/"
# results：收集每篇论文解析后的字典，稍后变成 DataFrame
results = []

# 按文件名排序后遍历；只处理 .pdf 后缀
for filename in sorted(os.listdir(papers_folder)):
    if filename.endswith(".pdf"):
        # 进度提示：当前正在处理哪份文件
        print(f"Processing: {filename}")
        # 拼出完整路径：文件夹 + 文件名
        pdf_path = os.path.join(papers_folder, filename)
        try:
            # 调 LLM 分析，得到字段字典
            data = analyze_paper(pdf_path)
            # 额外记下文件名，方便对照原始 PDF
            data["Filename"] = filename  # track which PDF each row came from
            results.append(data)
        except Exception as e:
            # 单篇失败不中断整批：打印错误后继续下一篇
            print(f"Error with {filename}: {e}")


# 把 list[dict] 收成表格，列名即 JSON 的 key
df = pd.DataFrame(results)

# ---------- 规范化「是否相关」列：统一成 Yes / No ----------
def standardize_relevant(val):
    # 模型有时会回 true/yes/1 等变体，这里映射到 Yes
    if str(val).lower() in ["true", "yes", "1"]:
        return "Yes"
    elif str(val).lower() in ["false", "no", "0"]:
        return "No"
    # 其它奇怪值时原逻辑默认 Yes
    return "Yes"

# 对整列 apply 规范化函数
df["Is Relevant"] = df["Is Relevant"].apply(standardize_relevant)

# ---------- 规范化相关性分数到约 1–10 量级 ----------
# 有的模型会回 0–1 小数，需要 *10

def standardize_score(val):
    try:
        score = float(val)
        if score <= 1.0:  # convert 0-1 scale to 0-10
            return round(score * 10, 1)
        return round(score, 1)
    except:
        # 无法转成数字时记为 None
        return None

df["Relevance Score"] = df["Relevance Score"].apply(standardize_score)

# 按分数降序排序，并重置行号
df = df.sort_values("Relevance Score", ascending=False).reset_index(drop=True)

# 先写出未着色的 Excel（后面再用 openpyxl 打开加样式）
output_file = "literature_survey_output.xlsx"
df.to_excel(output_file, index=False)

# ---------- 用 openpyxl 打开，准备颜色编码与排版 ----------
wb = load_workbook(output_file)
ws = wb.active

# 颜色：绿=高相关，黄=中等，红=低相关（ARGB 六位色值字符串保持原样）
green  = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
yellow = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")
red    = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")

# 读表头，定位「Relevance Score」列号（openpyxl 列从 1 起算）
headers = [cell.value for cell in ws[1]]
score_col = headers.index("Relevance Score") + 1

# 表头行去掉填充，保持默认白色
for cell in ws[1]:
    cell.fill = PatternFill(fill_type=None)

# 按每行分数给整行上色：>=8 绿，>=6 黄，否则红
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    score_cell = row[score_col - 1]
    try:
        score = float(score_cell.value)
        if score >= 8:
            fill = green
        elif score >= 6:
            fill = yellow
        else:
            fill = red
    except:
        # 分数异常时用黄色兜底
        fill = yellow
    for cell in row:
        cell.fill = fill

# 数据区：自动换行 + 顶端对齐，长文本更好读
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical="top")

# 按列内容估算列宽：短列略加 padding，长列上限 60
for col in ws.columns:
    max_length = 0
    col_letter = col[0].column_letter
    for cell in col:
        try:
            if cell.value:
                cell_length = len(str(cell.value))
                if cell_length > max_length:
                    max_length = cell_length
        except:
            pass
    if max_length < 15:
        adjusted_width = max_length + 4
    elif max_length < 50:
        adjusted_width = max_length + 2
    else:
        adjusted_width = 60
    ws.column_dimensions[col_letter].width = adjusted_width

# 保存最终格式化文件，并打印完成提示
wb.save(output_file)
print(f"Done! Check {output_file}")
